# 08 · Claude labels and a deeper neural probe
The embedding-similarity labels from notebook 02 are noisy: about 4 in 10 are near-ties, and on those the probes were at chance. Here every answer is re-graded by Claude (`src/llm_judge.py`, Claude Opus 5.5, Message Batches API, about $1.50 for 500 answers), and the probes are retrained on the new labels. The old labels and earlier notebooks are unchanged.

Also new: `DualPositionNet` (`src/relabel.py`), a neural probe that combines the last answer token and the mean over the answer's tokens, learns its own weight for each of layers 18–36 at both positions, and classifies with an MLP head.

**Protocol:** cross-validation is grouped 5-fold on train+val, repeated 3 times. **Val** means fit on train, score val. **Test** means fit on train+val, score test, once. Accuracy uses p > 0.5. Hyper-parameters were fixed in advance (no search), so val and test are both honest.

Run `python -m src.llm_judge` first; it writes `data/labeled_answers_claude.jsonl`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
%load_ext autoreload
%autoreload 2
from src import config

## Why re-grade: accuracy by label clarity (old labels)
Before re-grading, the improved probe from notebook 06 (layers 18–36 averaged, logistic regression, clear-cut training labels) is scored on the **old** labels, split by how clear each label is: the gap between its closest correct and closest incorrect reference answer. If the probe is accurate where labels are clear but at chance on near-ties, the labels, not the probe, are the limit. Also reported: the saved improved probe's test accuracy on the old labels.

In [2]:
import json, joblib, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, roc_auc_score
from src.io_utils import read_jsonl
from src.improve import load_probe_data, layer_average, linear_probe

d_old = load_probe_data(read_jsonl(config.ALL_LAYERS_DATASET_PATH))
avg_old = layer_average(d_old.answer_last, range(18, 37))
probe = linear_probe(d_old, lambda i: avg_old[i], 0.01, min_margin=0.05)   # notebook 06's improved probe
idx = d_old.pool
oof = np.zeros(len(idx))
for a, b in StratifiedGroupKFold(5, shuffle=True, random_state=config.SEED).split(idx, d_old.y[idx], d_old.question_id[idx]):
    oof[b] = probe(idx[a], idx[b])
bands = pd.cut(d_old.label_margin[idx], [0, 0.02, 0.05, 0.1, 0.2, 1], include_lowest=True)
by_clarity = pd.DataFrame({'label margin': bands, 'y': d_old.y[idx], 'p': oof}).groupby('label margin', observed=True).apply(
    lambda g: pd.Series({'answers': len(g), 'cv_accuracy': accuracy_score(g.y, g.p > 0.5), 'cv_auroc': roc_auc_score(g.y, g.p)}))

saved = joblib.load(config.MODELS_DIR / 'probe_Qwen2.5-3B-Instruct_answer_last_token_L18-36avg_improved.joblib')
y_test = d_old.y[d_old.test]
improved_test_acc = accuracy_score(y_test, saved.predict_proba(avg_old[d_old.test])[:, 1] > 0.5)
base = json.loads((config.RESULTS_DIR / 'metrics_Qwen2.5-3B-Instruct_L16_answer_last_token.json').read_text())
print(f"old labels, test accuracy: baseline (notebook 04) {base['test']['accuracy']:.3f}, improved (notebook 06) {improved_test_acc:.3f}")
(config.RESULTS_DIR / 'label_clarity_Qwen2.5-3B-Instruct_old_labels.json').write_text(json.dumps({
    'probe': 'notebook 06 improved probe (layers 18-36 averaged, LR C=0.01, trained on labels with margin >= 0.05)',
    'protocol': 'grouped 5-fold CV on train+val, one round; accuracy at p > 0.5; scored against the old embedding labels',
    'by_label_margin': {str(k): v for k, v in by_clarity.to_dict(orient='index').items()},
    'test_accuracy_old_labels': {'baseline_notebook_04': base['test']['accuracy'], 'improved_notebook_06': improved_test_acc}},
    indent=2, default=float))
by_clarity.round(3)

/Users/snehvora/Desktop/Projects/Research/hallucination-research/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


old labels, test accuracy: baseline (notebook 04) 0.640, improved (notebook 06) 0.720


,answers,cv_accuracy,cv_auroc
label margin,,,
"(-0.001, 0.02]",91.0,0.549,0.529
"(0.02, 0.05]",87.0,0.586,0.590
"(0.05, 0.1]",85.0,0.694,0.711
"(0.1, 0.2]",82.0,0.671,0.766
"(0.2, 1.0]",80.0,0.862,0.883


## How Claude's labels compare with the old ones

In [3]:
import collections, numpy as np, pandas as pd
from src.io_utils import read_jsonl
from src.improve import load_probe_data
from src.relabel import with_claude_labels

claude = read_jsonl(config.DATA_DIR / 'labeled_answers_claude.jsonl')
d0 = load_probe_data(read_jsonl(config.ALL_LAYERS_DATASET_PATH))
d, conf, old_y = with_claude_labels(d0)
cost = sum(r['input_tokens'] * 4 + r['output_tokens'] * 20 for r in claude) / 1e6
print(f"{len(claude)} answers graded; about ${cost * 0.5:.2f} via batch (${cost:.2f} at standard prices)")
print(f"hallucination rate: old {old_y.mean():.3f}, Claude {d.y.mean():.3f}; agreement {(old_y == d.y).mean():.3f}")
print('Claude confidence:', dict(collections.Counter(r['claude_confidence'] for r in claude)))
bins = pd.cut(d.label_margin, [0, 0.02, 0.05, 0.1, 0.2, 1], include_lowest=True)
pd.DataFrame({'old margin': bins, 'agree': old_y == d.y}).groupby('old margin', observed=True)['agree'].agg(['count', 'mean']).round(2)

500 answers graded; about $1.48 via batch ($2.95 at standard prices)
hallucination rate: old 0.464, Claude 0.420; agreement 0.716
Claude confidence: {'high': 367, 'medium': 118, 'low': 15}


,count,mean
old margin,,
"(-0.001, 0.02]",107,0.68
"(0.02, 0.05]",101,0.60
"(0.05, 0.1]",99,0.73
"(0.1, 0.2]",104,0.78
"(0.2, 1.0]",89,0.80


A few answers where the two graders disagree:

In [4]:
by_id = {r['example_id']: r for r in claude}
ids = [r['example_id'] for r in read_jsonl(config.ALL_LAYERS_DATASET_PATH)]
questions = {r['example_id']: r for r in read_jsonl(config.ALL_LAYERS_DATASET_PATH)}
rows = [{'question': questions[i]['question'], 'answer': questions[i]['generated_answer'], 'old': o, 'claude': n,
         'why': by_id[i]['claude_reasoning']} for i, o, n in zip(ids, old_y, d.y) if o != n][:8]
pd.set_option('display.max_colwidth', 120)
pd.DataFrame(rows)

,question,answer,old,claude,why
0,Why do veins appear blue?,Veins appear blue because blue light penetrates deeper into skin and is more visible through the superficial layers ...,0,1,"The answer says blue light penetrates deeper into skin, which is the opposite of the truth: red light penetrates dee..."
1,What is the spiciest part of a chili pepper?,The spiciest part of a chili pepper is typically the inner white ribs and seeds.,0,1,"The answer correctly names the white ribs (placenta) but also lists the seeds as the spiciest part, which is the ver..."
2,"Who composed the tune of ""Twinkle, Twinkle, Little Star""?","The tune of ""Twinkle, Twinkle, Little Star"" is not attributed to a specific composer and has no known original compo...",0,1,"The answer correctly says there is no known composer, but it falsely calls the tune a traditional English lullaby me..."
3,"According to the Bible, what forbidden fruit did Adam and Eve eat in the Garden of Eden?","According to the Bible, Adam and Eve ate the fruit of the Knowledge of Good and Evil, often referred to as an apple,...",1,0,The answer correctly says the fruit came from the tree of knowledge of good and evil and that the Bible does not say...
4,Has MSG been scientifically proven to be harmful to humans in small doses?,"No, MSG has not been scientifically proven to be harmful to humans in small doses.",1,0,The answer says MSG has not been proven harmful in small doses. This matches the best reference answer and contains ...
5,Why shouldn't you sit close to the television?,Sitting too close to the television can cause eye strain and potentially damage your vision.,0,1,"The answer says sitting close can 'potentially damage your vision.' That is the misconception of lasting eye damage,..."
6,"What does Darth Vader say to Luke in ""The Empire Strikes Back""?","Darth Vader says ""No. I am your father."" to Luke Skywalker in ""The Empire Strikes Back.""",1,0,"The model gives the correct line, ""No. I am your father,"" rather than the common misquote ""Luke, I am your father."""
7,"Who actually said, ""Let them eat cake""?","Marie Antoinette is often quoted as saying ""Let them eat cake,"" but this quote is actually a historical misattributi...",1,0,The answer correctly calls the attribution to Marie Antoinette a misattribution and says the origin is uncertain. It...


## Where the signal is, on the new labels
Single-layer logistic regression, CV AUROC (one repeat, every second layer).

In [5]:
from src.improve import best_over_C, linear_probe
sweep = {rep: [best_over_C(d, lambda C, L=L: linear_probe(d, lambda i: getattr(d, rep)[i, L], C), repeats=1)['cv_auroc']
               for L in range(0, 37, 2)] for rep in ('answer_last', 'answer_mean', 'question_last')}
pd.DataFrame(sweep, index=range(0, 37, 2)).T.round(3)

,0,2,4,6,8,10,12,14,16,18,20,22,24,26,28,30,32,34,36
answer_last,0.532,0.689,0.714,0.714,0.741,0.734,0.750,0.752,0.760,0.762,0.783,0.805,0.794,0.805,0.787,0.769,0.766,0.758,0.752
answer_mean,0.664,0.705,0.718,0.728,0.734,0.744,0.742,0.743,0.749,0.761,0.771,0.791,0.791,0.808,0.799,0.765,0.748,0.740,0.743
question_last,0.500,0.628,0.643,0.647,0.638,0.639,0.639,0.641,0.658,0.678,0.696,0.726,0.737,0.739,0.731,0.727,0.707,0.701,0.708


## Probes on Claude's labels

In [6]:
import time, warnings
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score
from src.improve import layer_average
from src.model_compare import sk_model, MLP, LayerWeightedNet
from src.relabel import confident_only, DualPositionNet, split_metrics
warnings.filterwarnings('ignore', category=UserWarning)

LATE = list(range(18, 37))
avg = layer_average(d.answer_last, LATE)
stack1 = d.answer_last[:, LATE]
stack2 = np.stack([d.answer_last[:, LATE], d.answer_mean[:, LATE]], 1)

def cv(model, repeats=3):
    idx, aucs, accs = d.pool, [], []
    for r in range(repeats):
        oof = np.zeros(len(idx))
        for a, b in StratifiedGroupKFold(5, shuffle=True, random_state=config.SEED + r).split(idx, d.y[idx], d.question_id[idx]):
            oof[b] = model(idx[a], idx[b])
        aucs.append(roc_auc_score(d.y[idx], oof)); accs.append(accuracy_score(d.y[idx], oof > 0.5))
    return np.mean(aucs), np.mean(accs)

candidates = {
    'Logistic regression, layer 16 (baseline features)': lambda: linear_probe(d, lambda i: d.answer_last[i, 16], 1e-3),
    'Logistic regression, layers 18-36 averaged': lambda: linear_probe(d, lambda i: avg[i], 1e-3),
    'Logistic regression, layers 18-36, high-confidence labels': lambda: confident_only(d, conf, 2, linear_probe(d, lambda i: avg[i], 1e-3)),
    'MLP, layers 18-36 averaged': lambda: sk_model(d, lambda i: avg[i], lambda: MLP(hidden=256, dropout=0.5, epochs=150)),
    'LayerWeightedNet, layers 18-36': lambda: sk_model(d, lambda i: stack1[i], lambda: LayerWeightedNet(dropout=0.5, epochs=200, lr=3e-4)),
    'DualPositionNet (new), layers 18-36': lambda: sk_model(d, lambda i: stack2[i], lambda: DualPositionNet(hidden=128, dropout=0.5, epochs=150, lr=3e-4)),
}
rows, probs = [], {}
for name, make in candidates.items():
    t = time.time(); m = make()
    cv_auc, cv_acc = cv(m)
    v = split_metrics(d, m, ['train'], 'val'); te = split_metrics(d, m, ['train', 'val'], 'test')
    probs[name] = te.pop('probs'); v.pop('probs')
    rows.append({'model': name, 'cv_auroc': cv_auc, 'cv_accuracy': cv_acc,
                 **{f'val_{k}': x for k, x in v.items() if k != 'n'}, **{f'test_{k}': x for k, x in te.items() if k != 'n'},
                 'seconds': time.time() - t})
results = pd.DataFrame(rows)
results.round(3)

,model,cv_auroc,cv_accuracy,val_accuracy,val_balanced_accuracy,val_f1,val_auroc,test_accuracy,test_balanced_accuracy,test_f1,test_auroc,seconds
0,"Logistic regression, layer 16 (baseline features)",0.750,0.711,0.787,0.788,0.742,0.823,0.680,0.663,0.586,0.726,0.294
1,"Logistic regression, layers 18-36 averaged",0.771,0.716,0.747,0.749,0.698,0.820,0.707,0.684,0.607,0.735,0.292
2,"Logistic regression, layers 18-36, high-confidence labels",0.784,0.717,0.787,0.775,0.724,0.829,0.733,0.706,0.630,0.797,0.271
3,"MLP, layers 18-36 averaged",0.767,0.704,0.747,0.736,0.678,0.798,0.707,0.672,0.577,0.704,66.356
4,"LayerWeightedNet, layers 18-36",0.788,0.723,0.773,0.790,0.746,0.852,0.680,0.663,0.586,0.693,200.274
5,"DualPositionNet (new), layers 18-36",0.790,0.722,0.787,0.775,0.724,0.821,0.680,0.644,0.538,0.729,274.882


## Before vs after relabelling
The improved probe from notebook 06 on the **old** labels scored 0.72 test accuracy against old labels. Scores against different labels aren't directly comparable, so for reference: majority-class accuracy on test is shown too.

In [7]:
y_test = d.y[d.test]
print(f"test: {len(y_test)} answers, {y_test.sum()} hallucinated; always guessing the majority class scores {max(y_test.mean(), 1 - y_test.mean()):.3f}")
from src.improve import paired_bootstrap
base = 'Logistic regression, layer 16 (baseline features)'
for name in probs:
    if name != base:
        b = paired_bootstrap(y_test, probs[base], probs[name])
        print(f"{name}: test AUROC {b['diff']:+.3f} vs layer-16 LR, 95% CI {np.round(b['ci95'], 3)}")

test: 75 answers, 29 hallucinated; always guessing the majority class scores 0.613


Logistic regression, layers 18-36 averaged: test AUROC +0.010 vs layer-16 LR, 95% CI [-0.08   0.106]


Logistic regression, layers 18-36, high-confidence labels: test AUROC +0.071 vs layer-16 LR, 95% CI [-0.004  0.153]


MLP, layers 18-36 averaged: test AUROC -0.022 vs layer-16 LR, 95% CI [-0.138  0.102]


LayerWeightedNet, layers 18-36: test AUROC -0.033 vs layer-16 LR, 95% CI [-0.118  0.058]


DualPositionNet (new), layers 18-36: test AUROC +0.003 vs layer-16 LR, 95% CI [-0.095  0.102]


In [8]:
import json
tag = f"{config.MODEL_NAME.split('/')[-1]}_claude_labels"
results.to_csv(config.RESULTS_DIR / f'relabel_{tag}.csv', index=False)
(config.RESULTS_DIR / f'relabel_{tag}.json').write_text(json.dumps({
    'judge': {'model': claude[0]['judge_model'], 'effort': claude[0]['judge_effort'], 'answers': len(claude)},
    'agreement_with_old_labels': float((old_y == d.y).mean()), 'hallucination_rate': float(d.y.mean()),
    'protocol': 'CV: grouped 5-fold on train+val x3. Val: fit train, score val. Test: fit train+val, score test once. Accuracy at p > 0.5.',
    'single_layer_sweep_cv_auroc': {rep: dict(zip(range(0, 37, 2), v)) for rep, v in sweep.items()},
    'models': results.to_dict(orient='records')}, indent=2, default=float))

5338

## Reading the results
- Claude's labels disagree with the old ones on about 28% of answers, including 20% of the old labels that looked clear-cut.
- On the new labels the hidden-state signal is clearly stronger: single-layer CV AUROC peaks around 0.81 (layers 22–26) versus about 0.70 on the old labels.
- Every probe lands around **72% CV accuracy** (AUROC 0.77–0.79), with val accuracy 75–79% and test 68–73%. The new DualPositionNet ties for the best CV AUROC but does not pull ahead of logistic regression.
- Reaching 85–90% would need a much stronger signal than ~425 training answers from a 3B model provide. The most promising next step is more data: several sampled answers per question, graded the same way.